# Interactive bump-on-tail dispersion relation
The sliders vary the two-Maxwellian equilibrium parameters. The upper-half-plane root shown is the strongest detected root within the search range.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from vlasov.linear_theory import dispersion_map, dominant_mode
from vlasov.plotting import plot_dispersion, plot_distribution
from vlasov.initial_conditions import maxwellian


In [2]:
def display_case(k=0.2, eps=0.1, T0=0.2, v0=3.8):
    wr = np.linspace(-1, 2, 180)
    wi = np.linspace(1e-3, 0.6, 140)
    omega, strength = dispersion_map(k, eps, T0, v0, wr, wi)
    mode = dominant_mode(k, eps, T0, v0, wr, wi)
    v = np.linspace(-6, 7, 400)
    distribution = ((1-eps)*maxwellian(v)
                    + eps*maxwellian(v - v0, T0))
    fig, (a,b) = plt.subplots(1, 2, figsize=(11,4))
    plot_distribution(v, distribution, ax=a)
    _, im = plot_dispersion(omega, strength, root=mode, ax=b)
    fig.colorbar(im,ax=b,label=r'$\log_{10}(1/|D|)$')
    if mode is not None:
        a.axvline(mode.real/k,ls='--',color='C1',label='Phase velocity')
        a.legend()
    fig.suptitle('Dominant growth = ' + (f'{mode.imag:.5f}' if mode is not None else 'not detected'))
    plt.tight_layout()
    plt.show()
widgets.interact(display_case,
    k=widgets.FloatSlider(value=.2,min=.1,max=.6,step=.05),
    eps=widgets.FloatSlider(value=.1,min=.01,max=.35,step=.01),
    T0=widgets.FloatSlider(value=.2,min=.05,max=1,step=.05),
    v0=widgets.FloatSlider(value=3.8,min=2,max=5,step=.1));


interactive(children=(FloatSlider(value=0.2, description='k', max=0.6, min=0.1, step=0.05), FloatSlider(value=…

## Optional animation and MP4 export
This animates varying drift velocity. Requires ffmpeg to save MP4; run the cell only when needed.


In [3]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate_drift(v0_values=np.linspace(2.5,4.5,25), k=.2, eps=.1, T0=.2):
    fig, (ax1,ax2)=plt.subplots(1,2,figsize=(11,4))
    wr=np.linspace(-1,2,150); wi=np.linspace(1e-3,.6,110)
    v=np.linspace(-6,7,400)
    def frame(i):
        ax1.clear(); ax2.clear()
        v0=v0_values[i]
        f=(1-eps)*maxwellian(v)+eps*maxwellian(v-v0,T0)
        plot_distribution(v,f,ax=ax1)
        omega,strength=dispersion_map(k,eps,T0,v0,wr,wi)
        mode=dominant_mode(k,eps,T0,v0,wr,wi)
        plot_dispersion(omega,strength,root=mode,ax=ax2)
        fig.suptitle(f'v0={v0:.2f}  gamma={mode.imag:.4f}' if mode is not None else f'v0={v0:.2f}: no mode detected')
    animation=FuncAnimation(fig,frame,frames=len(v0_values),interval=200)
    plt.close(fig)
    return animation


animation = animate_drift()
display(HTML(animation.to_jshtml()))
animation.save('dispersion.mp4', writer='ffmpeg', fps=5)
